# Zero-shot TSFM leaderboard on TSBench-Forge

**Question.** How do zero-shot pretrained time-series foundation models from the
[GIFT-Eval leaderboard](https://huggingface.co/spaces/Salesforce/GIFT-Eval) rank on
TSBench-Forge — a *live-data* benchmark whose truth windows post-date every model's
pretraining cutoff?

**Design.**
- Challenge set: deterministic 256-challenge draw from the live catalog
  (per-cadence context/horizon profiles, one light truth-preserving augmentation,
  unseen-fraction scoring weights).
- Models: every zero-shot TSFM that fits an 8 GB GPU with an adapter in
  `tsfm_adapters.py` (Chronos-Bolt tiny/small/base, Toto-2.0 4m/22m/313m, TiRex-35m)
  plus the frozen classical reference panel and both anti-gaming floors
  (seasonal-naive, context-parrot).
- Headline metric: **seasonal-naive-relative shifted geometric mean** of MASE and
  CRPS, normalised per source (the GIFT-Eval / BOOM convention). Arithmetic-mean
  metrics are reported alongside; rank gaps are tested with per-source Wilcoxon
  signed-rank tests.

**Reproducibility.** Everything is seeded; the challenge draw replays byte-identically
for a fixed data snapshot. Scores move as the live archive grows — record the data
snapshot date with any published numbers.


In [1]:
import os, sys, time, warnings
from pathlib import Path

import numpy as np

warnings.filterwarnings("ignore")
_cands = [Path.cwd(), Path.cwd() / "src", Path.cwd().parent / "src",
          Path(os.environ.get("TSBENCH_SRC", "/root/TSBench-Forge/src"))]
SRC = next(p for p in _cands if (p / "scraped_source.py").exists())
os.chdir(SRC); sys.path.insert(0, str(SRC))

import torch
from scraped_source import ScrapedLiveSource
from ingest import FreshBuffer
from challenges import build_live_challenges
from evaluate import leaderboard, probabilistic_panel, clears_floor, _per_challenge_scores
from tsfm_adapters import load_tsfm

SEED, N_CHALLENGES, POOL, MOTIF_LEN = 20260704, 256, 128, 608
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
print("data snapshot:", max(p.name for p in (SRC/'sources'/'data').glob('*/*.parquet')))

torch 2.5.1+cu121 | cuda: True
data snapshot: 2026-07-04.parquet


## 1. Challenge set

Breadth-balanced draw (equal weight per domain × DGP class × cadence), per-cadence
profile shapes, deterministic in the seed.

In [2]:
src = ScrapedLiveSource("sources/sources.yaml", "sources/data", min_series_length=MOTIF_LEN)
buf = FreshBuffer(src, pool_size=POOL, motif_len=MOTIF_LEN)
chs = build_live_challenges(buf, np.random.default_rng(SEED), N_CHALLENGES)

from collections import Counter
print("challenges:", len(chs))
print("shapes (ctx, h):", dict(sorted(Counter((c.meta["context_len"], c.meta["horizon"]) for c in chs).items())))
print("domains:", dict(Counter(c.meta["domain"] for c in chs)))
uf = np.array([c.meta["unseen_frac"] for c in chs])
print(f"unseen_frac: mean {uf.mean():.3f} | >0 on {int((uf>0).sum())} challenges")

challenges: 256
shapes (ctx, h): {(64, 8): 5, (128, 8): 5, (256, 14): 147, (512, 24): 52, (512, 48): 47}
domains: {'econ_fin': 36, 'transport': 35, 'nature': 34, 'healthcare': 31, 'energy': 41, 'sales': 44, 'web_cloudops': 35}
unseen_frac: mean 0.009 | >0 on 8 challenges


## 2. Zero-shot model registry

Each entry is attempted independently; unavailable weights/packages are skipped and
reported, so the notebook degrades gracefully on machines without every dependency.

In [3]:
REGISTRY = {
    "toto2-313m":        ("toto2",        {"model_name": "Datadog/Toto-2.0-313m"}),
    "toto2-22m":         ("toto2",        {"model_name": "Datadog/Toto-2.0-22m"}),
    "toto2-4m":          ("toto2",        {"model_name": "Datadog/Toto-2.0-4m"}),
    "tirex-35m":         ("tirex",        {}),
    "chronos-bolt-base": ("chronos-bolt", {"model_name": "amazon/chronos-bolt-base"}),
    "chronos-bolt-small":("chronos-bolt", {"model_name": "amazon/chronos-bolt-small"}),
    "chronos-bolt-tiny": ("chronos-bolt", {"model_name": "amazon/chronos-bolt-tiny"}),
}
models, skipped = {}, {}
for name, (kind, kw) in REGISTRY.items():
    try:
        fc = load_tsfm(kind, **kw)
        fc(chs[0].context, chs[0].meta)   # force weight download + one forecast
        models[name] = fc
        print(f"loaded {name}")
    except Exception as e:
        skipped[name] = f"{type(e).__name__}: {e}"
        print(f"SKIPPED {name}: {skipped[name][:100]}")
print(f"\n{len(models)} zero-shot models ready; {len(skipped)} skipped")

loaded toto2-313m


loaded toto2-22m


loaded toto2-4m


loaded tirex-35m


Loading weights:   0%|          | 0/269 [00:00<?, ?it/s]

...

Loading weights: 100%|██████████| 143/143 [00:00<00:00, 2711.29it/s]

loaded chronos-bolt-small


Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 101/101 [00:00<00:00, 3009.00it/s]

loaded chronos-bolt-tiny

7 zero-shot models ready; 0 skipped


## 3. Leaderboard

`MASE×SN` / `CRPS×SN` are per-source seasonal-naive-relative shifted geometric means
(1.0 = seasonal-naive parity; lower is better; ranking key = MASE×SN). `PCE` is the
probabilistic calibration error; `cov80` the empirical coverage of the nominal-80% band.

In [4]:
t0 = time.time()
board = leaderboard({**models, **probabilistic_panel()}, chs)
print(f"scored {len(board)} forecasters x {len(chs)} challenges in {time.time()-t0:.0f}s\n")
hdr = f"{'rank':>4} {'model':<20} {'MASExSN':>8} {'CRPSxSN':>8} {'MASE':>7} {'WQL':>7} {'CRPS':>7} {'PCE':>6} {'cov80':>6}"
print(hdr); print("-" * len(hdr))
for r in board:
    print(f"{r['rank']:>4} {r['model']:<20} {r['mase_rel']:>8.3f} {r['crps_rel']:>8.3f} "
          f"{r['mase']:>7.3f} {r['wql']:>7.3f} {r['crps']:>7.3f} {r['pce']:>6.3f} {r['coverage_80']:>6.2f}")

scored 13 forecasters x 256 challenges in 173s

rank model                 MASExSN  CRPSxSN    MASE     WQL    CRPS    PCE  cov80
---------------------------------------------------------------------------------
   1 strong                  0.718    0.873   1.365   1.015   0.626  0.083   0.85
   2 ewma                    0.746    0.853   1.414   1.022   0.622  0.078   0.84
   3 ar1                     0.788    0.872   1.637   0.978   0.613  0.074   0.84
   4 toto2-313m              0.820    0.456   1.138   0.511   0.379  0.018   0.78
   5 seasonal_naive          0.839    0.987   1.626   1.164   0.723  0.063   0.79
   6 toto2-22m               0.866    0.471   1.204   0.508   0.378  0.020   0.79
   7 toto2-4m                0.885    0.490   1.181   0.518   0.385  0.016   0.78
   8 drift                   0.886    0.957   1.656   1.136   0.709  0.069   0.82
   9 tirex-35m               0.975    0.535   1.166   0.518   0.382  0.029   0.78
  10 chronos-bolt-base       1.150    0.630   1.43

## 4. Floors

A zero-shot model only demonstrates skill if it beats **both** floors — the classical
seasonal-naive and the repetition (context-parrot) floor.

In [5]:
for name, fc in models.items():
    cf = clears_floor(fc, chs)
    print(f"{name:<20} clears={str(cf['clears']):<5} model={cf['candidate_score']:.3f} "
          f"floors={{sn: {cf['floors']['seasonal_naive']:.3f}, parrot: {cf['floors']['context_parrot']:.3f}}}")

toto2-313m           clears=True  model=1.138 floors={sn: 1.626, parrot: 2.745}


toto2-22m            clears=True  model=1.204 floors={sn: 1.626, parrot: 2.745}


toto2-4m             clears=True  model=1.181 floors={sn: 1.626, parrot: 2.745}


tirex-35m            clears=True  model=1.166 floors={sn: 1.626, parrot: 2.745}


chronos-bolt-base    clears=True  model=1.435 floors={sn: 1.626, parrot: 2.745}


chronos-bolt-small   clears=True  model=1.506 floors={sn: 1.626, parrot: 2.745}


chronos-bolt-tiny    clears=False model=1.762 floors={sn: 1.626, parrot: 2.745}


## 5. Are adjacent rank gaps real?

Per-source seasonal-naive-relative MASE, paired Wilcoxon signed-rank between every
pair of zero-shot models (heavy-tailed per-challenge scores make the t-test
under-powered here; Wilcoxon is the primary test). Lower-triangle: p-values;
upper: row-model win rate across sources.

In [6]:
from scipy import stats

sources = np.array([c.meta["source_id"] for c in chs])
groups = {s: np.flatnonzero(sources == s) for s in dict.fromkeys(sources)}
from evaluate import probabilistic
from score import seasonal_naive
nm, nc, w = _per_challenge_scores(probabilistic(seasonal_naive), chs, None)

rel = {}
for name, fc in models.items():
    ms, cs, _ = _per_challenge_scores(fc, chs, None)
    rel[name] = np.array([np.average(ms[i], weights=w[i]) / max(np.average(nm[i], weights=w[i]), 1e-8)
                          for i in groups.values()])
names = [r["model"] for r in board if r["model"] in models]
n = len(names)
print(f"per-source relative MASE over {len(groups)} sources\n")
print(" " * 20 + "".join(f"{m[:12]:>13}" for m in names))
for i, a in enumerate(names):
    row = f"{a[:18]:<20}"
    for j, b in enumerate(names):
        if i == j: row += f"{'—':>13}"
        elif i > j: row += f"{stats.wilcoxon(rel[a], rel[b]).pvalue:>13.3g}"
        else: row += f"{np.mean(rel[a] < rel[b]):>12.0%} "
    print(row)

per-source relative MASE over 25 sources

                       toto2-313m    toto2-22m     toto2-4m    tirex-35m chronos-bolt chronos-bolt chronos-bolt
toto2-313m                      —         52%          60%          68%          68%          68%          64% 
toto2-22m                   0.672            —         68%          56%          76%          76%          68% 
toto2-4m                     0.21       0.0626            —         44%          68%          68%          68% 
tirex-35m                   0.164        0.426        0.958            —         68%          72%          76% 
chronos-bolt-base          0.0367      0.00964       0.0318      0.00558            —         56%          52% 
chronos-bolt-small         0.0187       0.0105      0.00964      0.00558          0.3            —         52% 
chronos-bolt-tiny          0.0203       0.0115       0.0125      0.00251        0.458        0.653            —


## 6. Per-cadence-band breakdown

Where does each model earn its rank? Median per-challenge MASE by cadence band.

In [7]:
bands = np.array([c.meta["cadence"] or "?" for c in chs])
per_ch = {name: _per_challenge_scores(fc, chs, None)[0] for name, fc in models.items()}
uniq = [b for b in dict.fromkeys(bands)]
print(f"{'model':<20}" + "".join(f"{b:>12}" for b in uniq))
for name in names:
    print(f"{name:<20}" + "".join(f"{np.median(per_ch[name][bands == b]):>12.2f}" for b in uniq))
print(f"{'(n challenges)':<20}" + "".join(f"{int((bands == b).sum()):>12d}" for b in uniq))

model                      daily      hourly     sub-min      weekly      yearly     few-min   half-hour
toto2-313m                  0.84        0.82        0.68        0.01        0.86        0.44        0.94
toto2-22m                   0.83        0.82        0.68        0.01        0.86        0.36        0.94
toto2-4m                    0.82        0.82        0.69        0.02        0.86        0.57        0.93
tirex-35m                   0.85        0.80        0.67        0.01        0.88        0.65        0.83
chronos-bolt-base           0.85        0.85        0.68        0.07        0.85        0.69        0.95
chronos-bolt-small          0.85        0.86        0.68        0.07        0.89        0.77        1.03
chronos-bolt-tiny           0.86        0.87        0.68        0.06        0.83        0.75        0.88
(n challenges)               147          52          19           5           5          15          13


## 7. Reading the results

- **Ranking** — the top of the table is the "natural ranking" of zero-shot TSFMs on
  data none of them can have memorised: fresher than every pretraining cutoff, from a
  catalog that overlaps no standard corpus, with light truth-preserving augmentation.
- **Significance matrix** (§5) separates real gaps from leaderboard decoration:
  treat any pair with p > 0.05 (Bonferroni-correct for the number of pairs you read)
  as tied.
- **Scale trends** — with three Toto-2 sizes and three Chronos-Bolt sizes, §3 and §6
  show whether parameter count buys accuracy on live data, and in which cadence bands.
- **Caveats** — one data snapshot, one seed; per-source ratios punish source-specific
  blowups hard (a model can win most sources and still rank low — check §5 win rates
  against §3 ranks); classical panel rows use generic Gaussian bands, so compare their
  WQL/CRPS with that in mind.